# Version Extractor All 

This Notebook extracts all lines and der transcriptions from a given document in Escriptorium and saves it into a CSV file for further processing. 

Usage:

Run the notebook to imort the needed modules and define the functions. 
in the last cell - define the variables for the api_token (can be generated in eScriptorium for a certain user), documentid and filename and run the version_extractor_all function

--> version_extractor_all(documentid,token, api_url, filename)



In [ ]:
import requests
import json
import Levenshtein
import tablib 

In [ ]:
def type_check(documentid, token, apiURL, filename):
    """This function tests, if the parameters have the right type. It will be called in the version extractor function to check for the right types and 
    and raises errors, when needed"""
    if type(token) != str:
        raise TypeError ('Tokenparameter must be a string!')
    elif type(documentid) != int:
        raise TypeError ('DocumentID must be an integer!')
    elif type(apiURL) != str:
        raise TypeError ('apiURL must be a string!')
    elif type(filename) != str:
        raise TypeError ('Filename must be a string!')
    else: 
        print ('Parameters are all good!')

In [ ]:

def get_parts(token, documentid: int, page:int, api_url):
    """This function extracts all the parts (pages) of a document (documentid) and saves the parts in a list called parts """
    api_url = api_url
    url = api_url + str(documentid) + '/parts/?page=' + str(page)
    doc_response = requests.get(url, headers={'Authorization': 'Token ' + token})   
    json = doc_response.json()
    for result in json['results']:
        parts.append(result['pk'])
    page=page+1
    print('')
    print('Extracting pages ...')
    if json['next'] != None:
        get_parts(token,documentid,page, api_url)
    else:
        print('')
        print('All Done =^.^=')
        print('________________')
        print('            ')
        print('           ')
        print('           ')

In [ ]:

def get_lines(token, documentid, partsid, pagenum, api_url):
    """Gets the id of all the lines on a page and stores them in an empty list. If the statuscode 404 appears, it stopps and prints an ending message. Afterwards 
    the API points of the sites will be opened in a loop and all lineids are saved in a list"""   
    api_url = api_url
    url = api_url + str(documentid) + "/parts/" + str(partsid) + '/lines/?page=' + str(pagenum)
    doc_response = requests.get(url, headers={'Authorization': 'Token ' + token})   
    json = doc_response.json()
    
    if doc_response.status_code != 404:
        for line in json['results']:
            part = line['document_part']
            linenumber = line['pk']
            linenum.append([part,linenumber])
       
        pagenum = pagenum+1
        get_lines(token, documentid, partsid, pagenum, api_url)
    else:
         print('Extracting lines of of page id:' + '  ' + str(partsid) + ' ... ')

In [ ]:


def get_content(documentid):
    print('Extracting Transcriptions ....')
    for i in linenum:
        lines = []
        url = api_url + str(documentid) + "/parts/" + str(i[0]) + '/lines/' + str(i[1])
        doc_response = requests.get(url, headers={'Authorization': 'Token ' + token})   
        data = doc_response.json()
       
        for transcripts in data["transcriptions"]:
            content = transcripts["content"] # current content
            #clean_content=content.replace('\n', '')          
            lines.append(content)
            if transcripts["versions"] != []:
                versionlist =[]
                for versions in transcripts['versions']:
                    version = versions["data"]["content"]
                    update = versions["created_at"]
                    versionlist.append(version)
                    
                lines.append(versionlist)
                versioncount = len(versionlist)
                lines.insert(0, versioncount)
                #print(versionlist)
            else:
                lines.insert(0,["None"])

        #print(lines)
        linecontents.append(lines)
    print(' ')
    print('')
    print('All Done =^.^=')

In [ ]:
parts = []
linenum = []
linecontents = []


def version_extractor_all(documentid, token, apiURL, filename):
    """
    
    """
    api_url = apiURL
    print('Testing parameters ...')
    print('')
    type_check(documentid, token, apiURL, filename)
    print('')
    print('')
    print('')
    print('________START OF OPERATION________')
    print('')
    print('')
    get_parts(token, documentid, 1, api_url)
    for line in parts:
        get_lines(token,documentid,line, 1, api_url)
    print('')
    print('All Done =^.^=')
    print('________________')
    print('')
    print('')
    print('')
    get_content(documentid)
    #
    clean_list = list(filter(None,linecontents))
    
    DBLIST = tablib.Dataset(headers=['Current Version','Initial Version', 'Iterations' , 'Levenshtein Distance', 'Similarity Ratio'])
    print('Table is being prepared ...')
    for index in testlist:
        if index[0] == ['None']:
            current_version = index[1]
            initial_version = index[1]
            iterations = 0
            dist_calc = Levenshtein.distance(current_version, initial_version)
            similarity_ratio = Levenshtein.ratio(current_version, initial_version)
            DBLIST.append([current_version, initial_version, iterations, dist_calc, similarity_ratio])
        else:
            current_version = index[1]
            iterations = index[0]
            initial_version = index[2][-1] # oldest version 
            dist_calc = Levenshtein.distance(current_version, initial_version)
            similarity_ratio = Levenshtein.ratio(current_version, initial_version)
            DBLIST.append([current_version, initial_version, iterations, dist_calc, similarity_ratio])

    with open(filename,  'w', encoding="utf-8") as f:
        f.write(DBLIST.export('csv'))
    print('')
    print('')
    print('All Done =^.^=')
    print('')
    print('')
    print('Table saved to ' + filename)
    print('')
    print('')
    print('________END OF OPERATION________')
    

# Code in Action


In [ ]:
token ='' #set api token --> can be found in escriptorium django admin board 
documentid = #set documentid - must be integer
api_url = 'http://143.50.30.29:8080/api/documents/' 
filename = '' #set filename for csv

version_extractor_all(documentid,token, api_url, filename)
